# 🛡️ GigShield — MobileNetV3 Satellite Weather Classifier (CNN)

**Model:** MobileNetV3-Small (fine-tuned from ImageNet)
**Purpose:** Classify satellite imagery tiles into 4 weather categories
**Classes:** clear, light_rain, heavy_rain, flood_risk

This notebook:
1. Generates synthetic satellite-like training tiles (procedural generation)
2. Fine-tunes MobileNetV3-Small
3. Saves model weights
4. Tests classification on sample images

In [ ]:
!pip install torch torchvision numpy Pillow matplotlib -q

## Step 1: Generate Synthetic Satellite Tiles

Since real satellite data requires NASA POWER API access and labeling,
we generate realistic synthetic tiles using procedural noise patterns:
- **clear**: Blue sky gradients with sparse white dots (stars/cirrus)
- **light_rain**: Gray clouds with light texture
- **heavy_rain**: Dark gray/black with dense white streaks
- **flood_risk**: Very dark with blue tints (water reflections)

In [ ]:
import numpy as np
from PIL import Image, ImageFilter
import os

CLASSES = ['clear', 'light_rain', 'heavy_rain', 'flood_risk']
IMG_SIZE = 224
SAMPLES_PER_CLASS = 2000

def generate_clear(size=224):
    img = np.zeros((size, size, 3), dtype=np.uint8)
    # Sky blue gradient
    for y in range(size):
        r = int(100 + 80 * (y / size) + np.random.normal(0, 5))
        g = int(140 + 60 * (y / size) + np.random.normal(0, 5))
        b = int(200 + 30 * (y / size) + np.random.normal(0, 5))
        img[y, :] = np.clip([r, g, b], 0, 255)
    # Add sparse white clouds
    n_clouds = np.random.randint(3, 12)
    for _ in range(n_clouds):
        cx, cy = np.random.randint(0, size, 2)
        r = np.random.randint(5, 25)
        Y, X = np.ogrid[-cy:size-cy, -cx:size-cx]
        mask = X**2 + Y**2 <= r**2
        img[mask] = np.clip(img[mask] + 40, 0, 255)
    return Image.fromarray(img)

def generate_light_rain(size=224):
    base = np.random.randint(140, 180, (size, size, 3), dtype=np.uint8)
    base[:, :, 2] = np.clip(base[:, :, 2] + 20, 0, 255)
    noise = np.random.normal(0, 15, (size, size, 3)).astype(np.int16)
    img = np.clip(base.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    pil_img = Image.fromarray(img)
    return pil_img.filter(ImageFilter.GaussianBlur(radius=2))

def generate_heavy_rain(size=224):
    base = np.random.randint(50, 90, (size, size, 3), dtype=np.uint8)
    base[:, :, 2] = np.clip(base[:, :, 2] + 15, 0, 255)
    # Rain streaks
    n_streaks = np.random.randint(40, 100)
    for _ in range(n_streaks):
        x = np.random.randint(0, size)
        y_start = np.random.randint(0, size // 2)
        length = np.random.randint(20, 60)
        for dy in range(length):
            yy = min(y_start + dy, size - 1)
            xx = min(x + dy // 4, size - 1)
            base[yy, xx] = np.clip(base[yy, xx] + 60, 0, 255)
    return Image.fromarray(base)

def generate_flood_risk(size=224):
    base = np.random.randint(20, 50, (size, size, 3), dtype=np.uint8)
    base[:, :, 2] = np.clip(base[:, :, 2] + 40, 0, 255)
    # Water reflection patches
    n_patches = np.random.randint(5, 15)
    for _ in range(n_patches):
        cx, cy = np.random.randint(0, size, 2)
        r = np.random.randint(15, 50)
        Y, X = np.ogrid[-cy:size-cy, -cx:size-cx]
        mask = X**2 + Y**2 <= r**2
        base[mask, 0] = np.clip(base[mask, 0] - 10, 0, 255)
        base[mask, 1] = np.clip(base[mask, 1] - 5, 0, 255)
        base[mask, 2] = np.clip(base[mask, 2] + 30, 0, 255)
    return Image.fromarray(base)

GENERATORS = {
    'clear': generate_clear,
    'light_rain': generate_light_rain,
    'heavy_rain': generate_heavy_rain,
    'flood_risk': generate_flood_risk,
}

for cls in CLASSES:
    os.makedirs(f'satellite_tiles/{cls}', exist_ok=True)
    gen = GENERATORS[cls]
    for i in range(SAMPLES_PER_CLASS):
        img = gen(IMG_SIZE)
        img.save(f'satellite_tiles/{cls}/{i:04d}.png')
    print(f'Generated {SAMPLES_PER_CLASS} tiles for {cls}')

print(f'\n✅ Total: {len(CLASSES) * SAMPLES_PER_CLASS} synthetic satellite tiles')

## Step 2: Create DataLoader

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

full_dataset = datasets.ImageFolder('satellite_tiles', transform=train_transform)

# Split 80/20
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_ds, val_ds = torch.utils.data.random_split(full_dataset, [train_size, val_size])
val_ds.dataset.transform = val_transform

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=32, num_workers=2)

print(f'Train: {len(train_ds)}, Val: {len(val_ds)}')
print(f'Classes: {full_dataset.classes}')

## Step 3: Define & Fine-tune MobileNetV3

In [ ]:
from torchvision import models

class WeatherVerifier(nn.Module):
    def __init__(self, num_classes=4, pretrained=True):
        super().__init__()
        if pretrained:
            self.model = models.mobilenet_v3_small(weights='IMAGENET1K_V1')
        else:
            self.model = models.mobilenet_v3_small(weights=None)
        # Freeze all except last 2 conv blocks
        for param in self.model.parameters():
            param.requires_grad = False
        for param in self.model.features[-2:].parameters():
            param.requires_grad = True
        # Replace classifier
        in_features = self.model.classifier[-1].in_features
        self.model.classifier[-1] = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.model(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

model = WeatherVerifier(pretrained=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=5e-5)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'Parameters: {trainable:,} trainable / {total:,} total ({100*trainable/total:.1f}%)')

In [ ]:
best_acc = 0
for epoch in range(20):
    model.train()
    train_loss, correct, total_samples = 0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total_samples += labels.size(0)
    train_acc = 100 * correct / total_samples

    model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            val_correct += predicted.eq(labels).sum().item()
            val_total += labels.size(0)
    val_acc = 100 * val_correct / val_total

    print(f'Epoch {epoch+1:2d} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {train_acc:.1f}% | Val Acc: {val_acc:.1f}%')
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'cnn_model.pt')

print(f'\n✅ Best validation accuracy: {best_acc:.1f}%')
print('✅ Model saved to cnn_model.pt')

## Step 4: Test Classification

In [ ]:
import matplotlib.pyplot as plt

model.load_state_dict(torch.load('cnn_model.pt', map_location=device))
model.eval()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, cls in enumerate(CLASSES):
    img = GENERATORS[cls](224)
    tensor = val_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1)[0]
        pred_idx = probs.argmax().item()
    axes[i].imshow(img)
    axes[i].set_title(f'True: {cls}\nPred: {CLASSES[pred_idx]} ({probs[pred_idx]:.0%})')
    axes[i].axis('off')
plt.tight_layout()
plt.savefig('cnn_test_results.png', dpi=100)
plt.show()
print('\n✅ Test results saved to cnn_test_results.png')

## Step 5: Download Model

Download `cnn_model.pt` and place in `backend/ml/cnn_verify/`

In [ ]:
try:
    from google.colab import files
    files.download('cnn_model.pt')
except:
    print('Not in Colab. File saved locally.')
    print('Copy cnn_model.pt -> backend/ml/cnn_verify/cnn_model.pt')